# Intelligent Document Understanding (IDU) System
### End-to-End Deep Learning Walkthrough: Layout Analysis, Key Information Extraction, Table Parsing & DocVQA

This notebook demonstrates the complete IDU pipeline from raw scanned document/PDF to structured key-value entities, tabular data, and visual question answering.

In [ ]:
import sys
import os
import json
import pandas as pd
from PIL import Image

# Ensure project root is on PYTHONPATH
sys.path.append(os.path.abspath(".."))

from idu_core.pipeline import DocumentUnderstandingPipeline
from idu_core.config import config

print(f"IDU System Version: {config.version}")
print(f"Compute Device: {config.model.device.upper()}")

## 1. Initialize Unified Pipeline
We initialize the `DocumentUnderstandingPipeline` with the auto-selected OCR engine and Multimodal Transformer.

In [ ]:
pipeline = DocumentUnderstandingPipeline(ocr_engine="auto")
sample_path = "../samples/sample_invoice.png"
print(f"Processing: {sample_path}")

## 2. Process Document & Extract Structured Data
The pipeline handles Preprocessing, OCR, Document Classification, 2D Spatial Multimodal Token Classification, Table Extraction, and Entity Linking.

In [ ]:
result = pipeline.process(sample_path, generate_visualizations=True)

print("=== Document Understanding Summary ===")
print(f"Document Classification: {result.document_type} ({result.type_confidence * 100:.1f}% confidence)")
print(f"Pages Processed:         {result.total_pages}")
print(f"Tokens Identified:        {result.total_tokens}")
print(f"Entities Extracted:       {result.total_entities}")
print(f"Tables Detected:          {result.total_tables}")
print(f"Execution Time:           {result.execution_time_seconds:.3f} seconds")

## 3. Extracted Entities & Normalization
Inspect the normalized structured key-values (Total Amounts, Dates, Invoice Numbers, Vendor Names).

In [ ]:
entities_summary = result.to_dict()["extracted_entities"]
print(json.dumps(entities_summary, indent=2))

## 4. Extracted Tables to Pandas DataFrame
Any tables present in the document are automatically reconstructed into 2D tabular formats.

In [ ]:
for p in result.pages:
    for idx, table in enumerate(p.tables, 1):
        print(f"Table #{idx}:")
        df = table.to_dataframe()
        display(df)

## 5. Document Visual Question Answering (DocVQA)
Query the document using natural language.

In [ ]:
questions = [
    "What is the invoice number?",
    "What is the invoice date?",
    "Who is the vendor?",
    "What is the total amount due?"
]

for q in questions:
    ans = pipeline.ask(q, result)
    print(f"Q: {q}")
    print(f"A: {ans['answer']} (Confidence: {ans['confidence'] * 100:.1f}%)\n")